In [ ]:
# import np # 네이버 뉴스를 검색해서 제목과 링크를 가져오는 모듈
import requests
from bs4 import BeautifulSoup
import pandas as pd

# naver_nl_ai_페이지.csv 파일에 있는 뉴스 링크로 가서 기사 내용을 가져와서,
# naver_nl_ai_페이지_article.csv를 생성하는 예제
# naver_nl_ai_페이지_article.csv 열 이름 => title, href, detail

url = 'https://n.news.naver.com/mnews/article/421/0008824068?sid=101'

headers = {
	'User-Agent' : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
	'Referer' : 'https://www.naver.com/'
}

response = requests.get(url, headers=headers)

try:
	response.raise_for_status()
	
except Exception as e:
	print(f"예외 발생 {e}")


In [ ]:
soup = BeautifulSoup(response.text, 'lxml')

article = soup.select_one("#dic_area")

print(article.text)

In [ ]:
def get_naver_news_article(url):
	"""네이버 기사에서 내용을 추출하여 반환"""
	url = 'https://n.news.naver.com/mnews/article/421/0008824068?sid=101'

	headers = {
		'User-Agent' : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
		'Referer' : 'https://www.naver.com/'
	}

	response = requests.get(url, headers=headers)

	try:
		response.raise_for_status()
		
	except Exception as e:
		print(f"예외 발생 {e}")

	soup = BeautifulSoup(response.text, 'lxml')

	article = soup.select_one("#dic_area")

	return article.text

In [ ]:
import np
import pandas as pd

# csv 파일에서 가져와서 모든 기사내용을 가져옴
keyword = 'ai'

for page in range(1, 5 + 1):
	df = pd.read_csv(f"naver_nl_{keyword}_{page}.csv", encoding='utf-8')

	articles = [] # 기사 목록
	for href in df['href']:
		# 주어진 url에 맞는 기사 내용을 가져와서 기사 목록에 추가
		articles.append(np.get_naver_news_article(href).strip())

	# 기존 df에 'article'열을 추가
	df['article'] = articles

	df.to_csv(f'naver_nl_{keyword}_{page}_article.csv', encoding='utf-8-sig', index=False)

In [ ]:
import np
import pandas as pd

# csv 파일에서 가져와서 모든 기사내용을 가져옴
keyword = 'ai'
total_df = pd.DataFrame({
	'title' : [],
	'href' : [],
	'article' : []
})
for page in range(1, 5 + 1):
	df = pd.read_csv(f"naver_nl_{keyword}_{page}.csv", encoding='utf-8')

	articles = [] # 기사 목록
	for href in df['href']:
		# 주어진 url에 맞는 기사 내용을 가져와서 기사 목록에 추가
		articles.append(np.get_naver_news_article(href).strip())

	# 기존 df에 'article'열을 추가
	df['article'] = articles
	total_df = pd.concat([df, total_df], axis=0)

total_df.to_csv(f'naver_nl_{keyword}_total_article.csv', encoding='utf-8-sig', index=False)




In [ ]:
print(len(total_df))

In [1]:
# 금융 뉴스 20개(2페이지까지)를 가져와서 csv파일 1개 저장
# title, href, atricle
import pandas as pd
import np

# 검색어
keyword = 'ai'
# 최대 페이지수
max_page = 2
# 전체 데이터 프레임 total_df 선언
total_df = pd.DataFrame({
	'title' : [], 
	'href' : [], 
	'article' : []
})
# 반복문
for page in range(1, max_page + 1):
	# 현재 페이지, 검색어에 맞는 뉴스 검색 페이지에서 뉴스 제목과 링크로 구성된 DataFrame df 가져옴
	df = np.get_naver_news_list(keyword, (page - 1) * 10 + 1)
	# 기사목록 리스트
	articles = []

	# 반복문으로 df에 있는 뉴스 페이지 기사 내용을 가져옴
	for href in df['href']:
		# 가져온 뉴스 기사 내용을 뉴스 기사 목록에 이어 붙임
		# 링크가 없는 경우 기사가 없음
		article = np.get_naver_news_article(href) if href else None
		articles.append(article)

	# df에 'article' 열에 뉴스 기사 목록 추가
	df['article'] = articles
	# total_df에 df를 이어 붙임
	total_df = pd.concat([total_df, df], axis=0)

# 링크가 없거나 기사가 없으면 제거
total_df = total_df.dropna()

# csv파일에 저장
total_df.to_csv(f'naver_nl_{keyword}_article.csv', encoding="utf-8-sig", index=False)	